In [ ]:
import xml.etree.ElementTree as ET
from collections import defaultdict, Counter
import re
import pickle

class HandTransitionAnalyzer:
    """
    Analyzes MSCX files to extract hand transition patterns for Grade 1 piano exercises.
    """
    
    def __init__(self):
        # Main dictionary: {time_signature: {num_bars: {pattern: count}}}
        self.transition_patterns = defaultdict(lambda: defaultdict(Counter))
        
        # Track exercise boundaries (key or time signature changes)
        self.current_exercise = []
        self.exercise_start = 0
        
    def parse_mscx(self, filepath):
        """Parse a single MSCX file and extract exercises"""
        tree = ET.parse(filepath)
        root = tree.getroot()
        
        # Find all measures
        measures = root.findall('.//Measure')
        
        exercises = []
        current_exercise = []
        last_key = None
        last_time = None
        
        for i, measure in enumerate(measures):
            # Check for key or time signature changes (new exercise)
            key_sig = measure.find('.//KeySig')
            time_sig = measure.find('.//TimeSig')
            
            # Get current key and time
            current_key = key_sig.get('accidental') if key_sig is not None else last_key
            current_time = None
            if time_sig is not None:
                sigN = time_sig.find('sigN')
                sigD = time_sig.find('sigD')
                if sigN is not None and sigD is not None:
                    current_time = f"{sigN.text}/{sigD.text}"
            
            # If this is first measure or key/time changed, start new exercise
            is_new_exercise = False
            if i == 0:
                is_new_exercise = True
            elif (current_key != last_key and current_key is not None) or \
                 (current_time != last_time and current_time is not None):
                is_new_exercise = True
            
            if is_new_exercise and current_exercise:
                # Save previous exercise
                exercises.append(current_exercise)
                current_exercise = []
            
            current_exercise.append(measure)
            last_key = current_key if current_key is not None else last_key
            last_time = current_time if current_time is not None else last_time
        
        # Add last exercise
        if current_exercise:
            exercises.append(current_exercise)
        
        return exercises
    
    def analyze_hand_movement(self, measure, staff_id=1):
        """
        Analyze which hand is playing in a measure.
        Returns 'R' (right hand), 'L' (left hand), or 'RL'/'LR' for transitions.
        """
        voice = measure.find('.//voice')
        if voice is None:
            return 'R'  # Default to right hand if no voice
        
        # Check stem directions to determine hand
        chords = voice.findall('.//Chord')
        if not chords:
            return 'R'  # Rest measure
            
        # Track hand changes within measure
        current_hand = None
        hand_changes = []
        
        for chord in chords:
            stem_dir = chord.get('StemDirection')
            # In piano, right hand typically has stems up, left hand stems down
            hand = 'R' if stem_dir == 'up' else 'L' if stem_dir == 'down' else 'R'
            
            if current_hand is None:
                current_hand = hand
            elif hand != current_hand:
                hand_changes.append((hand, chord))
                current_hand = hand
        
        if not hand_changes:
            return current_hand  # Single hand throughout measure
        else:
            # Determine transition pattern within measure
            first_hand = 'R'  # Default
            if chords:
                first_stem = chords[0].get('StemDirection')
                first_hand = 'R' if first_stem == 'up' else 'L'
            
            last_hand = 'R'  # Default
            if chords:
                last_stem = chords[-1].get('StemDirection')
                last_hand = 'R' if last_stem == 'up' else 'L'
            
            return f"{first_hand}{last_hand}"  # 'RL' or 'LR'
    
    def get_time_signature(self, measure):
        """Extract time signature from measure"""
        time_sig = measure.find('.//TimeSig')
        if time_sig is not None:
            sigN = time_sig.find('sigN')
            sigD = time_sig.find('sigD')
            if sigN is not None and sigD is not None:
                return f"{sigN.text}/{sigD.text}"
        return None
    
    def get_key_signature(self, measure):
        """Extract key signature from measure"""
        key_sig = measure.find('.//KeySig')
        if key_sig is not None:
            accidental = key_sig.get('accidental')
            if accidental:
                # Convert to key name (simplified)
                acc_val = int(accidental)
                if acc_val > 0:
                    return f"{acc_val} sharps"
                elif acc_val < 0:
                    return f"{abs(acc_val)} flats"
                else:
                    return "C major/A minor"
        return "C major/A minor"  # Default
    
    def extract_hand_patterns(self, filepath):
        """Main method to extract hand transition patterns from a file"""
        exercises = self.parse_mscx(filepath)
        
        for exercise in exercises:
            # Get time signature from first measure of exercise
            time_sig = self.get_time_signature(exercise[0])
            if not time_sig:
                continue  # Skip if no time signature
            
            num_bars = len(exercise)
            
            # Analyze hand pattern for each measure
            hand_pattern = []
            for measure in exercise:
                hand = self.analyze_hand_movement(measure)
                hand_pattern.append(hand)
            
            # Convert to tuple for dictionary key
            pattern_tuple = tuple(hand_pattern)
            
            # Update counts
            self.transition_patterns[time_sig][num_bars][pattern_tuple] += 1
            
            # Optional: Print for debugging
            print(f"Exercise: {time_sig}, {num_bars} bars, Pattern: {pattern_tuple}")
    
    def get_transition_dictionary(self):
        """Return the compiled transition dictionary"""
        return self.transition_patterns
    
    def print_summary(self):
        """Print a summary of all found patterns"""
        for time_sig, bar_counts in self.transition_patterns.items():
            print(f"\n=== Time Signature: {time_sig} ===")
            for num_bars, patterns in bar_counts.items():
                print(f"  {num_bars} bars:")
                for pattern, count in patterns.most_common(10):  # Show top 10
                    print(f"    {pattern}: {count}")

def create_hand_pattern_database(mscx_files, output_file='hand_patterns.pkl'):
    """
    Quick function to create and save hand pattern database
    
    Args:
        mscx_files: List of paths to MSCX files
        output_file: Path to save the pickle file
    """
    analyzer = HandTransitionAnalyzer()
    
    for file in mscx_files:
        print(f"Processing {file}...")
        analyzer.extract_hand_patterns(file)
    
    # Save to pickle
    with open(output_file, 'wb') as f:
        pickle.dump(analyzer.get_transition_dictionary(), f)
    
    print(f"\n✅ Database saved to {output_file}")
    analyzer.print_summary()
    
    return analyzer

# Usage:
files = ["example_exercises/grade1_1_20.mscx", "example_exercises/grade1_21_45.mscx", "example_exercises/grade1_46_55.mscx"]
analyzer = create_hand_pattern_database(files, "grade1_patterns.pkl")

Processing example_exercises/grade1_1_20.mscx...
Exercise: 4/4, 4 bars, Pattern: ('R', 'R', 'R', 'R')
Exercise: 3/4, 4 bars, Pattern: ('R', 'R', 'R', 'R')
Exercise: 2/4, 6 bars, Pattern: ('R', 'R', 'R', 'R', 'R', 'R')
Exercise: 4/4, 8 bars, Pattern: ('R', 'R', 'R', 'R', 'R', 'R', 'R', 'R')
Exercise: 3/4, 8 bars, Pattern: ('R', 'R', 'R', 'R', 'R', 'R', 'R', 'R')
Exercise: 4/4, 4 bars, Pattern: ('R', 'R', 'R', 'R')
Exercise: 2/4, 6 bars, Pattern: ('R', 'R', 'R', 'R', 'R', 'R')
Exercise: 4/4, 8 bars, Pattern: ('R', 'R', 'R', 'R', 'R', 'R', 'R', 'R')
Exercise: 3/4, 5 bars, Pattern: ('R', 'R', 'R', 'R', 'R')
Exercise: 2/4, 6 bars, Pattern: ('R', 'R', 'R', 'R', 'R', 'R')
Exercise: 4/4, 8 bars, Pattern: ('R', 'R', 'R', 'R', 'R', 'R', 'R', 'R')
Exercise: 2/4, 6 bars, Pattern: ('R', 'R', 'R', 'R', 'R', 'R')
Exercise: 4/4, 4 bars, Pattern: ('R', 'R', 'R', 'R')
Exercise: 3/4, 4 bars, Pattern: ('R', 'R', 'R', 'R')
Exercise: 4/4, 4 bars, Pattern: ('R', 'R', 'R', 'R')
Exercise: 2/4, 6 bars, Pattern:

AttributeError: Can't get local object 'HandTransitionAnalyzer.__init__.<locals>.<lambda>'